# 📈 Unified Growth Explorer

**One notebook to see all growth rates, feel their differences, and understand why they dictate algorithm performance.**

---

## The Hook
Why do we care about asymptotic growth? Consider sorting 1 million books...

## 1. Concept Intuition

Imagine you're sorting books.

- **Linear (n):** You check every book once. 100 books → 100 checks. Fair.
- **Logarithmic (log n):** You keep *halving* the search space (like binary search in a sorted shelf). 1,000,000 books → ~20 checks. Magic.
- **Quadratic (n²):** You compare every book with every other book. 1,000 books → 1,000,000 comparisons. Painful.
- **Exponential (2ⁿ):** You try every possible *subset* of books. 30 books → over a billion subsets. Impossible.

### The crucial insight

For small n, they all feel similar. But algorithms don't live in small-n land — they live where n grows. And when n grows, the growth *rate* is the only thing that matters.

$n = 100$:
| Growth | Operations |
|--------|------------|
| log n | ~7 |
| n | 100 |
| n log n | ~664 |
| n² | 10,000 |
| 2ⁿ | 1.27 × 10³⁰ |

## The Pattern
**What did you notice?**
(Look at how the curves separate...)

## 2. Visual Representation

Let's see all growth rates on the same plot.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

n = np.linspace(1, 50, 500)

growth_rates = {
    'O(1)':       np.ones_like(n),
    'O(log n)':   np.log2(n),
    'O(n)':       n,
    'O(n log n)': n * np.log2(n),
    'O(n²)':      n**2,
    'O(2ⁿ)':      2**n,
}

colors = ['#10b981', '#06b6d4', '#6366f1', '#8b5cf6', '#f43f5e', '#ef4444']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Linear scale — shows how exponential dominates
ax = axes[0]
for (name, values), color in zip(growth_rates.items(), colors):
    ax.plot(n, values, label=name, color=color, linewidth=2)
ax.set_ylim(0, 2500)
ax.set_title('Growth Rates (linear scale)', fontsize=13, fontweight='bold')
ax.set_xlabel('n (input size)')
ax.set_ylabel('Operations')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Log scale — reveals structure across magnitudes
ax = axes[1]
for (name, values), color in zip(growth_rates.items(), colors):
    ax.plot(n, values, label=name, color=color, linewidth=2)
ax.set_yscale('log')
ax.set_title('Growth Rates (log scale)', fontsize=13, fontweight='bold')
ax.set_xlabel('n (input size)')
ax.set_ylabel('Operations (log)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Interactive Experiment

### Growth rate comparator
Use the slider to set `n_max` and watch how the curves separate.

In [ ]:
from ipywidgets import interact, IntSlider, FloatSlider

@interact(
    n_max=IntSlider(value=20, min=5, max=100, step=5, description='n_max'),
    coefficient=FloatSlider(value=1.0, min=0.1, max=10.0, step=0.1, description='coeff')
)
def growth_comparator(n_max, coefficient):
    n = np.linspace(1, n_max, 500)
    
    fig, ax = plt.subplots(figsize=(9, 5))
    
    rates = {
        'O(log n)':   coefficient * np.log2(n),
        'O(n)':       coefficient * n,
        'O(n log n)': coefficient * n * np.log2(n),
        'O(n²)':      coefficient * n**2,
    }
    colors = ['#10b981', '#6366f1', '#8b5cf6', '#f43f5e']
    
    for (name, values), color in zip(rates.items(), colors):
        ax.plot(n, values, label=name, color=color, linewidth=2)
    
    ax.set_title(f'Growth rates up to n = {n_max} (coeff = {coefficient:.1f})', 
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('n')
    ax.set_ylabel('Operations')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

### Dominance explorer
Watch lower-order terms become irrelevant as n grows.

In [ ]:
@interact(
    a=FloatSlider(value=1.0, min=0.1, max=10.0, step=0.1, description='a (n² coeff)'),
    b=FloatSlider(value=50.0, min=0, max=200, step=5, description='b (n coeff)'),
    c=FloatSlider(value=500.0, min=0, max=2000, step=50, description='c (constant)'),
    n_max=IntSlider(value=100, min=10, max=500, step=10, description='n_max')
)
def dominance_explorer(a, b, c, n_max):
    n = np.linspace(1, n_max, 500)
    
    f_full = a * n**2 + b * n + c
    f_quad = a * n**2
    f_linear = b * n
    f_const = np.full_like(n, c)
    
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    
    # Term breakdown
    axes[0].plot(n, f_full, label=f'{a}n² + {b}n + {c}', color='#6366f1', linewidth=2.5)
    axes[0].plot(n, f_quad, label=f'{a}n²', color='#f97316', linewidth=2, linestyle='--')
    axes[0].plot(n, f_linear, label=f'{b}n', color='#10b981', linewidth=1.5, linestyle=':')
    axes[0].axhline(y=c, color='#94a3b8', linestyle=':', label=f'constant = {c}')
    axes[0].set_title('Term breakdown', fontsize=12, fontweight='bold')
    axes[0].legend(fontsize=9)
    axes[0].grid(True, alpha=0.3)
    
    # Fraction from each term
    safe_full = np.maximum(f_full, 1e-10)
    axes[1].fill_between(n, 0, f_quad/safe_full * 100, alpha=0.6, color='#f97316', label='n² share')
    axes[1].fill_between(n, f_quad/safe_full * 100, (f_quad + f_linear)/safe_full * 100, 
                         alpha=0.6, color='#10b981', label='n share')
    axes[1].fill_between(n, (f_quad + f_linear)/safe_full * 100, 100, 
                         alpha=0.6, color='#94a3b8', label='constant share')
    axes[1].set_ylim(0, 100)
    axes[1].set_title('Percentage contribution of each term', fontsize=12, fontweight='bold')
    axes[1].set_ylabel('% of total')
    axes[1].legend(fontsize=9)
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

### Scaling animation: watch growth diverge

## 3. Mathematical Formulation

### Big-O: Asymptotic Upper Bound

We say $f(n) = O(g(n))$ if there exist constants $c > 0$ and $n_0$ such that:

$$f(n) \leq c \cdot g(n) \quad \text{for all } n \geq n_0$$

This means: **eventually**, $g(n)$ grows at least as fast as $f(n)$ (up to a constant).

### Asymptotic Dominance

A function $g$ **dominates** $f$ if:

$$\lim_{n \to \infty} \frac{f(n)}{g(n)} = 0$$

The hierarchy:

$$1 \prec \log n \prec n \prec n\log n \prec n^2 \prec 2^n \prec n!$$

### Limits Intuition

"Approaching infinity" means: no matter how large a number you name, $n$ will *eventually* exceed it. We care about what happens in that regime — where only the dominant term survives.

## 4. Code Implementation

### Asymptotic dominance: watching lower terms vanish

In [ ]:
# Watch how 3n² + 100n + 999 converges to n² behavior
n = np.linspace(1, 200, 1000)

f_full = 3 * n**2 + 100 * n + 999
f_dominant = 3 * n**2

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Absolute values
axes[0].plot(n, f_full, label='3n² + 100n + 999', color='#6366f1', linewidth=2)
axes[0].plot(n, f_dominant, label='3n²', color='#f97316', linewidth=2, linestyle='--')
axes[0].set_title('Absolute: curves converge', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Ratio approaches 1
ratio = f_full / f_dominant
axes[1].plot(n, ratio, color='#10b981', linewidth=2)
axes[1].axhline(y=1, color='gray', linestyle=':', alpha=0.5)
axes[1].set_title('Ratio → 1 (lower terms vanish)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('f(n) / 3n²')
axes[1].set_xlabel('n')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Computational cost simulation: MEASURE actual time
import time

def measure_time(func, n_values):
    """Measure wall-clock time for a function across input sizes."""
    times = []
    for n in n_values:
        start = time.perf_counter()
        func(n)
        elapsed = time.perf_counter() - start
        times.append(elapsed)
    return np.array(times)

# Three algorithms with different growth rates
def linear_search(n):
    arr = np.arange(n)
    return n - 1 in arr  # O(n) scan

def nested_loop(n):
    count = 0
    for i in range(n):
        for j in range(n):
            count += 1
    return count  # O(n²)

def binary_search(n):
    arr = np.arange(n)
    target = n - 1
    lo, hi = 0, n - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if arr[mid] == target: return mid
        elif arr[mid] < target: lo = mid + 1
        else: hi = mid - 1

sizes = np.array([100, 500, 1000, 2000, 3000, 4000, 5000])

t_log = measure_time(binary_search, sizes)
t_lin = measure_time(linear_search, sizes)
t_quad = measure_time(nested_loop, sizes)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(sizes, t_log * 1e6, 'o-', label='Binary search ~ O(log n)', color='#10b981', linewidth=2)
ax.plot(sizes, t_lin * 1e6, 's-', label='Linear search ~ O(n)', color='#6366f1', linewidth=2)
ax.plot(sizes, t_quad * 1e6, '^-', label='Nested loop ~ O(n²)', color='#f43f5e', linewidth=2)
ax.set_xlabel('Input size (n)', fontsize=12)
ax.set_ylabel('Time (μs)', fontsize=12)
ax.set_title('Measured Runtime: Theory Meets Reality', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

fig, ax = plt.subplots(figsize=(9, 5))

lines = {}
growth_funcs = {
    'O(log n)':   (lambda n: np.log2(n), '#10b981'),
    'O(n)':       (lambda n: n, '#6366f1'),
    'O(n log n)': (lambda n: n * np.log2(n), '#8b5cf6'),
    'O(n²)':      (lambda n: n**2, '#f43f5e'),
}

for name, (func, color) in growth_funcs.items():
    line, = ax.plot([], [], label=name, color=color, linewidth=2)
    lines[name] = (line, func)

ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlabel('n', fontsize=12)
ax.set_ylabel('Operations', fontsize=12)

def init():
    ax.set_xlim(1, 10)
    ax.set_ylim(0, 100)
    return [l for l, _ in lines.values()]

def animate(frame):
    n_max = 10 + frame * 2
    n = np.linspace(1, n_max, 500)
    y_max = 0
    for name, (line, func) in lines.items():
        y = func(n)
        line.set_data(n, y)
        y_max = max(y_max, y[-1])
    ax.set_xlim(1, n_max)
    ax.set_ylim(0, y_max * 1.1)
    ax.set_title(f'Growth divergence (n up to {n_max})', fontsize=13, fontweight='bold')
    return [l for l, _ in lines.values()]

ani = FuncAnimation(fig, animate, init_func=init, frames=50, interval=120, blit=False)
plt.close(fig)
HTML(ani.to_jshtml())

## 6. Tool Exploration

Understanding `np.linspace` and vectorized operations.

In [ ]:
# Inspect how linspace works
samples = np.linspace(0, 10, 6)
print(f'linspace(0, 10, 6) = {samples}')
print(f'Step size: {samples[1] - samples[0]}')
print(f'Shape: {samples.shape}')
print()

# Vectorization: no loops needed
print(f'samples ** 2    = {samples ** 2}')
print(f'np.log2(samples + 1) = {np.log2(samples + 1)}')
print()

# Boolean operations on arrays
print(f'samples > 5     = {samples > 5}')
print(f'Count > 5: {np.sum(samples > 5)}')
print()

# Inspect an Axes object after plotting
fig, ax = plt.subplots()
ax.plot(samples, samples**2)
plt.close(fig)

print(f'Axes xlim: {ax.get_xlim()}')
print(f'Axes ylim: {ax.get_ylim()}')
print(f'Number of lines: {len(ax.lines)}')
print(f'Line data: {ax.lines[0].get_data()}')

## 7. Reflection Questions

1. Why does `O(n log n)` appear so often in sorting algorithms? What property of comparison-based sorting forces this lower bound?

2. If you have an O(n²) algorithm and you double the input size, by what factor does the runtime increase? What about O(2ⁿ)?

3. In the dominance explorer, set `b = 1000` and `c = 0`. For what value of `n` does the n² term first exceed the n term? How does this change with different `a` values?

4. Why do we use Big-O notation instead of exact operation counts? What information do we lose, and why is that acceptable?

5. An algorithm runs in $5n + 10000$ steps. Another runs in $n^2$ steps. For what input sizes is the first algorithm actually slower? What does this tell you about the practical limits of asymptotic analysis?

---

*You now see growth. Next: understand structure. → `Structure_and_Transformations/`*

## 🔬 Break-It Lab
(Exercises merged from earlier structure)

In [ ]:
import numpy as np
from math101.testing import check_answer, check_complexity, difficulty_banner, score_report, predict_then_run
from math101.tracker import mark_complete, record_score

results = []

## 🟢 Warm-Up: Recall & Predict

## 🟡 Challenge: Apply & Analyze

## 📊 Results

In [ ]:
score_report(results)
record_score('Growth_and_Complexity', 'exercises', sum(results), len(results))

## The Feynman Technique
Explain [this concept] in plain English to someone who has never seen it. Write it in the cell below. Then check: did you use any jargon you can't define from scratch? If yes, go back.

*(Write your explanation here...)*

## Review
- **Q:** 
- **A:** 

- **Q:** 
- **A:** 

## The Takeaway
> **The one thing to carry forward is:** *(Write the insight, not the formula)*